# Process all abstracts into serialized graphs

This notebook keeps the batch loop in `graphicalizer.batch`. It processes every matching abstract in `ABSTRACT_FOLDER`, embeds each `node_context.summary`, and saves one NetworkX graph per abstract under `GRAPH_FOLDER`.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sentence_transformers import SentenceTransformer
from graphicalizer import (
    ExtractionDensityConfig,
    Graphicalizer,
    GraphicalizerConfig,
    NetworkXGraphStore,
    NodeContextConfig,
    NodeEmbeddingConfig,
    load_ontology,
    process_abstract_folder,
)

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
ASSETS_ROOT = PROJECT_ROOT / 'assets'
ABSTRACT_FOLDER = ASSETS_ROOT / 'abstracts' / 'pubmed'
ABSTRACT_PATTERN = '*.txt'
GRAPH_FOLDER = PROJECT_ROOT / 'outputs' / 'graphs'
GRAPH_ID_PREFIX = 'pubmed'
CONTINUE_ON_ERROR = True

ONTOLOGY_PATH = (
    ASSETS_ROOT / 'ontologies' / 'entity_ontology_microbiology_assembled.yaml'
    if (ASSETS_ROOT / 'ontologies' / 'entity_ontology_microbiology_assembled.yaml').exists()
    else ASSETS_ROOT / 'ontologies' / 'entity_ontology_microbiology.yaml'
)
LLM_PROVIDER = 'openai'  # choose 'openai' or 'ollama'
LLM_MODEL = 'gpt-4o-mini' if LLM_PROVIDER == 'openai' else 'gemma4:12b-mlx'
LLM_OPTIONS = (
    {'max_output_tokens': 16384}
    if LLM_PROVIDER == 'openai'
    else {'num_ctx': 32768, 'num_predict': -1}
)
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
EMBEDDING_MODEL = SentenceTransformer(EMBEDDING_MODEL_NAME)

ontology = load_ontology(ONTOLOGY_PATH)
density = ExtractionDensityConfig(
    entities_per_word=0.05,
    relations_per_entity=1.5,
    minimum_entity_fraction=0.75,
    density_retries=2,
)
config = GraphicalizerConfig(
    provider=LLM_PROVIDER,
    model=LLM_MODEL,
    extraction_density=density,
    node_context=NodeContextConfig(),
    prompt_template_path=ASSETS_ROOT / 'prompts' / 'graphicalizer_prompt_template.yaml',
    prompt_snapshot_path=PROJECT_ROOT / 'outputs' / 'prompts' / 'ontology-aware-graphicalizer-0.4.0.yaml',
    context_policy='all_nodes',
)
graphicalizer = Graphicalizer.from_provider(
    ontology,
    config,
    options=LLM_OPTIONS,
    embedding_model=EMBEDDING_MODEL,
    embedding_config=NodeEmbeddingConfig(model_id=EMBEDDING_MODEL_NAME),
)
graph_store = NetworkXGraphStore(GRAPH_FOLDER)

In [ ]:
batch = process_abstract_folder(
    ABSTRACT_FOLDER,
    graphicalizer,
    graph_store,
    pattern=ABSTRACT_PATTERN,
    graph_id_prefix=GRAPH_ID_PREFIX,
    continue_on_error=CONTINUE_ON_ERROR,
)

print('Discovered:', batch.discovered)
print('Processed:', batch.processed)
print('Failed:', batch.failed)
print('Manifest:', batch.manifest_path)
if batch.failures:
    print('Failures:')
    for failure in batch.failures:
        print(failure)